In [30]:
import pandas as pd
import numpy as np

SCORING_PATH = PROC_DIR / "scoring_with_amenity_scores.csv"
test_enc = pd.read_csv(SCORING_PATH)

print("Loaded scoring:", test_enc.shape)

Loaded scoring: (8997, 123)


In [31]:
# ---- features (same logic as training) ----
DROP_COLS = ["ubid", "time_window_tag", TARGET_COL]
FEATURES = [c for c in test_enc.columns if c not in DROP_COLS]

print("Total FEATURES:", len(FEATURES))

Total FEATURES: 120


In [32]:
cat_cols = [
    c for c in FEATURES
    if test_enc[c].dtype == "object"
]

cat_idx = [FEATURES.index(c) for c in cat_cols]

print("Categorical features:", len(cat_cols))

Categorical features: 11


In [33]:
X_test = test_enc[FEATURES].copy()

# VERY IMPORTANT: CatBoost cannot see NaN in categorical features
for c in cat_cols:
    X_test[c] = X_test[c].astype("object").fillna("__MISSING__")

print("X_test shape:", X_test.shape)

X_test shape: (8997, 120)


In [35]:
# X_test already created as test_enc[FEATURES].copy()

# Force all object/string columns to be treated as categorical
cat_cols = [c for c in FEATURES if X_test[c].dtype == "object"]

# IMPORTANT: fill NaN in categorical cols (CatBoost cannot handle NaN in cats)
for c in cat_cols:
    X_test[c] = X_test[c].astype("object").fillna("__MISSING__")

# Also ensure numeric columns are numeric (optional safety)
num_cols = [c for c in FEATURES if c not in cat_cols]
for c in num_cols:
    X_test[c] = pd.to_numeric(X_test[c], errors="coerce")

print("Categorical cols:", len(cat_cols))
print("Example categorical cols:", cat_cols[:10])

Categorical cols: 11
Example categorical cols: ['type_main', 'type_sub', 'city', 'state', 'mrkt_name', 'submrkt_name', 'trade_area_label', 'state_metro', 'msa_ring', 'market']


In [36]:
from catboost import Pool

test_pool = Pool(
    data=X_test,
    cat_features=cat_cols
)

pred_cb = final_cb.predict(test_pool)

test_enc["pred_target_catboost_post"] = pred_cb
print(test_enc["pred_target_catboost_post"].describe())

count    8997.000000
mean       -0.106509
std         0.029530
min        -0.382591
25%        -0.125578
50%        -0.105795
75%        -0.085285
max         0.161986
Name: pred_target_catboost_post, dtype: float64


In [37]:
OUT_PATH = PROC_DIR / "scoring_with_amenity_scores.csv"
test_enc.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

Saved: /Users/linda/RiceDatathon_2026_Finance/Data/Processed/scoring_with_amenity_scores.csv


In [38]:
print("Train target describe:")
print(y.describe())

print("\nTrain target quantiles:")
print(y.quantile([0.01,0.05,0.25,0.5,0.75,0.95,0.99]))

Train target describe:
count    19457.000000
mean        -0.051733
std          0.132668
min         -0.741112
25%         -0.130536
50%         -0.063088
75%          0.014964
max          1.944841
Name: target, dtype: float64

Train target quantiles:
0.01   -0.332192
0.05   -0.235275
0.25   -0.130536
0.50   -0.063088
0.75    0.014964
0.95    0.162124
0.99    0.298784
Name: target, dtype: float64


In [39]:
print(test_enc["time_window_tag"].value_counts(dropna=False))
print(test_enc["time_window_label"].value_counts(dropna=False).head(10))

time_window_tag
post    4512
pre     4485
Name: count, dtype: int64
time_window_label
2022_to_2025       4512
2015_to_2020Feb    4485
Name: count, dtype: int64


In [40]:
cols = [c for c in test_enc.columns if "pred_target" in c]
print(cols)

print("\nCorrelation between predictions:")
print(test_enc[cols].corr())

['pred_target_catboost_post']

Correlation between predictions:
                           pred_target_catboost_post
pred_target_catboost_post                        1.0
